# Create TROPESS AIRS-OMI Day Range Plots

## Overview
This notebook will allow you to download [TROPESS](https://tes.jpl.nasa.gov/tropess) AIRS-OMI data and then use that data to create daily plots for a given date range.  
The following suite of plots will be created from standard and summary data products:

| Species | Standard Product | Summary Product |
| :------ | :------ | :------ |
| Ozone (O3) | TRPSDL2O3AIRSOMIFS | TRPSYL2O3AIRSOMIFS |

In the table above, the "short name" of the data products is listed (e.g., "TRPSYL2O3AIRSOMIFS").  Short names are assigned by a NASA DAAC (in this case, the GES-DISC)
for each data products as a way to lookup or refer to a product with out using the product's full long name (e.g., "TROPESS AIRS-Aqua and OMI-Aura L2 Ozone for Reanalysis Stream, Summary Product V1").  You'll see these short names in the code below when there are blocks looping through the different products.

## Requirements

This notebook uses NCDISC's [earthaccess](https://www.earthdata.nasa.gov/news/blog/earthaccess-earth-science-data-simplified) Python package to retrieve requested data.  It requires a [NASA Earthdata login](https://urs.earthdata.nasa.gov/).  Please make sure you have one before attempting to run this notebook.  Also make sure you've setup a `.netrc` file in your home directory with your NASA Earthata login information.  You can use the follow steps to generate that file, if needed:

```sh
cd ~
touch .netrc
echo "machine urs.earthdata.nasa.gov login uid_goes_here password password_goes_here" > .netrc
chmod 0600 .netrc
```

## Import Libraries 

Standard (i.e., available through pip) libraries `datetime` and `os` are imported below.  You will also need to clone or install the `tropessplots` library if you haven't already.  It is available at [https://github.com/NASA-TROPESS/tropessplots](https://github.com/NASA-TROPESS/tropessplots).

In [ ]:
import datetime as dt
import earthaccess
import os

from tropessplots.io.airs_omi import read_l2summary, read_l2standard
from tropessplots.website_plots.airs_omi import plot_daily_overview

%load_ext autoreload
%autoreload 2

## Setup Global Variables

Please edit the variables in the next block as needed.  We'll do some checks and conversions after those variables are
set to reformat the date so we can use it in later blocks and to create any directories that don't exist.  We also have a 
list of all the data products we want to plot.

In [ ]:
# The date you want to plot in YYYYMMDD format.
START_DATE = '20250901'
END_DATE= '20250905'

# Where you want to store the data you download.
DOWNLOAD_DIRECTORY = '/tmp/download'

# Where you want to store the plot outputs.
PLOT_DIRECTORY = '/tmp/download'

# A list of the species you want to plot.  All species are listed here
# by default.  Edits to this list (additions or subtractions) will be 
# reflected in the SHORT_NAME_LIST variable below.  
SPECIES_ARRAY = ['O3']


In [ ]:
# The short names of the data products you want to plot.  These are automatically
# generated from SPECIES_ARRAY above.  Do not manually edit this list.
SPECIES_DICT = {}

if 'O3' in SPECIES_ARRAY:
    SPECIES_DICT['O3'] = {}
    SPECIES_DICT['O3']['standard']      = 'TRPSDL2O3AIRSOMIFS'
    SPECIES_DICT['O3']['summary']       = 'TRPSYL2O3AIRSOMIFS'

# Formatting dates for use later
start_date_object = dt.datetime.strptime(START_DATE, '%Y%m%d')
end_date_object = dt.datetime.strptime(END_DATE, '%Y%m%d')
start_date_time = start_date_object.strftime('%Y-%m-%d 00:00:00')
end_date_time = end_date_object.strftime('%Y-%m-%d 23:59:59')

# Create directories if they don't already exist
os.makedirs(DOWNLOAD_DIRECTORY, exist_ok=True)
os.makedirs(PLOT_DIRECTORY, exist_ok=True)

## Search for and Download Data

 Now that we are setup, we're going to start our work by downloading data from NASA Earthdata. We'll first search for files on the shot name, start and end dates, and version number. Once the service returns with a list of matching files, we'll download them."

### Function to Find Files
This is a function we'll keep using as we plot data.  We'll define it here and use it later, in case you are trying to plot multiple species.  It will take the short name, start date, and end date as inputs and retun a list of files.

In [ ]:
def findFiles(short_name, start_date_time, end_date_time):
    earthaccess.login(strategy='netrc')
    results = earthaccess.search_data(count=-1, short_name=short_name, temporal=(start_date_time, end_date_time))
    if len(results) == 0:
        return
    print('Downloading %s' % short_name)
    files = earthaccess.download(results, DOWNLOAD_DIRECTORY)
    
    return files

### Generate the plots
Here, we'll do the work we need to do to generate a plot for each species for the given time range.  First we will search for data, generating a list of files and downloading them.  Then, we'll read in any standard and summary files to generate the plots.

In [ ]:
for species in SPECIES_DICT.keys():
    # Find and download the files the files
    
    SPECIES_DICT[species]['standard_file_list'] = findFiles(SPECIES_DICT[species]['standard'], start_date_time, end_date_time)
    SPECIES_DICT[species]['summary_file_list'] = findFiles(SPECIES_DICT[species]['summary'], start_date_time, end_date_time)
    figure_file = PLOT_DIRECTORY + '/TROPESS_AIRS-OMI_' + species + '_' + START_DATE + '-' + END_DATE + '.png'

    local_standard_list = [DOWNLOAD_DIRECTORY + '/' + item.split('/')[-1] for item in SPECIES_DICT[species]['standard_file_list']]
    if 'summary' in SPECIES_DICT[species]:
        local_summary_list = [DOWNLOAD_DIRECTORY + '/' + item.split('/')[-1] for item in SPECIES_DICT[species]['summary_file_list']]
    
    # Read data
    l2summary = read_l2summary(files=local_summary_list,
                                verbose=0)
    l2standard = read_l2standard(files=local_standard_list,
                                verbose=0)
    # Run plotting routine
    plot_daily_overview(l2summary=l2summary,
                        l2standard=l2standard,
                        file_out=figure_file)
    print('Produced plot %s' % figure_file)